# GP Regression — Effect of Length-scale

This notebook demonstrates **GP regression** with a squared-exponential (RBF) kernel on synthetic data,
highlighting how the length-scale hyperparameter $\ell$ controls the smoothness of the fit.

| Plot | Description |
|---|---|
| **Plot 0** | Raw data with noise error bars |
| **Plot 1** | GP fit — correct $\ell = 1.0$ |
| **Plot 2** | GP fit — too short $\ell = 0.3$ (over-fitting) |
| **Plot 3** | GP fit — too long $\ell = 3.0$ (under-fitting) |
| **Plot 4** | Random posterior samples for all three $\ell$ values |

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from gp_utils import gp_posterior_se
from kernels import squared_exponential

In [ ]:
np.random.seed(42)

N = 12
X_train = np.random.uniform(-5, 5, size=N)
X_train.sort()

ell_true = 1.0
sf_true  = 1.0
sn_true  = 0.1

K = squared_exponential(X_train, X_train, lengthscale=ell_true, variance=sf_true ** 2)
f = np.random.multivariate_normal(np.zeros(N), K)
y = f + np.random.normal(0, sn_true, size=N)

X_test = np.linspace(-5.5, 5.5, 200)

# Conditional latent sample (conditioned on the noiseless signal f)
K_xx   = squared_exponential(X_train, X_train, lengthscale=ell_true, variance=sf_true ** 2)
K_xxs  = squared_exponential(X_train, X_test,  lengthscale=ell_true, variance=sf_true ** 2)
K_xsxs = squared_exponential(X_test,  X_test,  lengthscale=ell_true, variance=sf_true ** 2)
L_true = np.linalg.cholesky(K_xx + 1e-10 * np.eye(N))
alpha_t = np.linalg.solve(L_true.T, np.linalg.solve(L_true, f))
mu_fs   = K_xxs.T @ alpha_t
cov_fs  = K_xsxs - K_xxs.T @ np.linalg.solve(K_xx, K_xxs)
f_star  = np.random.multivariate_normal(mu_fs, cov_fs + 1e-10 * np.eye(len(X_test)))

# GP posteriors for the three length-scales
ell_values = [1.0, 0.3, 3.0]
posts = {ell: gp_posterior_se(X_train, y, X_test, ell, sf_true, sn_true) for ell in ell_values}

# Shared y-axis range
sns.set_theme(style="whitegrid", palette="husl")
sns.set_context("notebook", font_scale=1.1)

y_min = np.min(y - sn_true)
y_max = np.max(y + sn_true)
for ell in ell_values:
    mu, cov = posts[ell]
    sd = np.sqrt(np.diag(cov))
    y_min = min(y_min, np.min(mu - 2 * sd))
    y_max = max(y_max, np.max(mu + 2 * sd))
y_min = min(y_min, np.min(f_star))
y_max = max(y_max, np.max(f_star))
margin = 0.1 * (y_max - y_min)
y_min -= margin
y_max += margin

## Plot 0: Observed Data

In [ ]:
fig0, ax0 = plt.subplots(figsize=(10, 6))
ax0.errorbar(
    X_train, y, yerr=sn_true,
    fmt='o', ms=10, elinewidth=2, capsize=5, label="Observed data", color='steelblue'
)
ax0.set_title("Observed Data with Noise Error Bars", fontsize=14, fontweight='bold')
ax0.set_xlabel("Input (x)", fontsize=12)
ax0.set_ylabel("Output (y)", fontsize=12)
ax0.set_xlim(-5.5, 5.5)
ax0.set_ylim(y_min, y_max)
ax0.legend(loc='best', frameon=True, shadow=True)
ax0.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Plot 1: GP Regression — Correct Length-scale ($\ell = 1.0$)

In [ ]:
fig1, ax1 = plt.subplots(figsize=(10, 6))
mu, cov = posts[1.0]
sd = np.sqrt(np.diag(cov))
ax1.fill_between(X_test, mu - 2*sd, mu + 2*sd, alpha=0.3, label="95% confidence interval")
ax1.plot(X_test, mu, lw=2.5, label="GP mean prediction")
ax1.plot(X_test, f_star, "--", lw=2, alpha=0.7, label="Conditional latent sample")
ax1.errorbar(
    X_train, y, yerr=sn_true,
    fmt='o', ms=8, elinewidth=2, capsize=4, label="Observed data", color='steelblue'
)
ax1.set_title("GP Regression: ℓ = 1.0 (correct length-scale)", fontsize=14, fontweight='bold')
ax1.set_xlabel("Input (x)", fontsize=12)
ax1.set_ylabel("Output (y)", fontsize=12)
ax1.set_xlim(-5.5, 5.5)
ax1.set_ylim(y_min, y_max)
ax1.legend(loc='best', frameon=True, shadow=True)
ax1.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Plot 2: GP Regression — Too Short ($\ell = 0.3$, over-fitting)

In [ ]:
fig2, ax2 = plt.subplots(figsize=(10, 6))
mu, cov = posts[0.3]
sd = np.sqrt(np.diag(cov))
ax2.fill_between(X_test, mu - 2*sd, mu + 2*sd, alpha=0.3, label="95% confidence interval")
ax2.plot(X_test, mu, lw=2.5, label="GP mean prediction")
ax2.errorbar(
    X_train, y, yerr=sn_true,
    fmt='o', ms=8, elinewidth=2, capsize=4, label="Observed data", color='steelblue'
)
ax2.set_title("GP Regression: ℓ = 0.3 (too short)", fontsize=14, fontweight='bold')
ax2.set_xlabel("Input (x)", fontsize=12)
ax2.set_ylabel("Output (y)", fontsize=12)
ax2.set_xlim(-5.5, 5.5)
ax2.set_ylim(y_min, y_max)
ax2.legend(loc='best', frameon=True, shadow=True)
ax2.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Plot 3: GP Regression — Too Long ($\ell = 3.0$, under-fitting)

In [ ]:
fig3, ax3 = plt.subplots(figsize=(10, 6))
mu, cov = posts[3.0]
sd = np.sqrt(np.diag(cov))
ax3.fill_between(X_test, mu - 2*sd, mu + 2*sd, alpha=0.3, label="95% confidence interval")
ax3.plot(X_test, mu, lw=2.5, label="GP mean prediction")
ax3.errorbar(
    X_train, y, yerr=sn_true,
    fmt='o', ms=8, elinewidth=2, capsize=4, label="Observed data", color='steelblue'
)
ax3.set_title("GP Regression: ℓ = 3.0 (too long)", fontsize=14, fontweight='bold')
ax3.set_xlabel("Input (x)", fontsize=12)
ax3.set_ylabel("Output (y)", fontsize=12)
ax3.set_xlim(-5.5, 5.5)
ax3.set_ylim(y_min, y_max)
ax3.legend(loc='best', frameon=True, shadow=True)
ax3.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Plot 4: Random Posterior Samples — All Three Length-scales

In [ ]:
fig4, ax4 = plt.subplots(figsize=(10, 6))
colors = ['#e74c3c', '#3498db', '#2ecc71']  # red, blue, green
labels = ['ℓ = 1.0 (correct)', 'ℓ = 0.3 (too short)', 'ℓ = 3.0 (too long)']
for ell, color, label in zip(ell_values, colors, labels):
    mu, cov = posts[ell]
    sample = np.random.multivariate_normal(mu, cov + 1e-10 * np.eye(len(X_test)))
    ax4.plot(X_test, sample, lw=2.5, alpha=0.8, color=color, label=label)
ax4.errorbar(
    X_train, y, yerr=sn_true,
    fmt='o', ms=8, elinewidth=2, capsize=4, color='steelblue', label="Observed data"
)
ax4.set_title("Random Posterior Samples from Different Length-scales", fontsize=14, fontweight='bold')
ax4.set_xlabel("Input (x)", fontsize=12)
ax4.set_ylabel("Output (y)", fontsize=12)
ax4.set_xlim(-5.5, 5.5)
ax4.set_ylim(y_min, y_max)
ax4.legend(loc='best', frameon=True, shadow=True)
ax4.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()